# Scene Classification v3 — Google Colab + Drive

## Cấu trúc thư mục Google Drive
```
MyDrive/
└── Colab Notebooks/
    └── processing_data/          ← DATA_DIR (ảnh đầu vào)
        ├── bedroom/
        ├── coast/
        └── ...                   (15 lớp cảnh)
└── scene_ckpt/                   ← CKPT_DIR (checkpoint tự động)
    ├── labels.npz
    ├── pca_raw_descriptors.npz   ← raw descriptor để vẽ biểu đồ PCA
    ├── pca.pkl
    ├── pca_elbow.png
    ├── kmeans.pkl
    ├── kmeans_elbow.png
    ├── gmm.pkl
    ├── features_train.npz
    ├── features_test.npz
    ├── scalers_train.pkl
    ├── clfs.pkl
    ├── weights.pkl
    ├── results.pkl
    ├── confusion_matrix.png
    └── accuracy_comparison.png
```



---
## Cell 1 — Cài đặt thư viện

In [ ]:
!pip install -q opencv-python-headless scikit-image scikit-learn scipy tqdm

---
## Cell 2 — Import

In [ ]:
import os, glob, gc, pickle, warnings, shutil
from pathlib import Path
from dataclasses import dataclass
from functools import lru_cache

for _k in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_k, "1")

import numpy as np
import cv2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 110

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.svm import LinearSVC, SVC # Added SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier # Added RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from scipy.optimize import minimize
from skimage.feature import graycomatrix, graycoprops

warnings.filterwarnings('ignore')
print('✅ Import OK')

---
## Cell 3 — Config  ⚙️  (chỉnh tại đây)

> **Lưu ý:** `pca_dim` và `vlad_k` ban đầu để `None` — sẽ được xác định sau khi xem biểu đồ elbow ở Cell 10 và Cell 13.

In [ ]:
@dataclass
class Config:
    # ── Google Drive paths ──────────────────────────────────────────────────
    drive_root : Path = Path("/content/drive/MyDrive")
    data_dir   : Path = Path("/content/drive/MyDrive/Colab Notebooks/processing_data")
    ckpt_dir   : Path = Path("/content/drive/MyDrive/scene_ckpt")
    # Thư mục tạm local (nhanh hơn Drive khi đọc/ghi liên tục)
    local_ckpt : Path = Path("/content/scene_ckpt_local")

    img_exts     : tuple = ("*.jpg","*.jpeg","*.png","*.bmp","*.webp")
    test_size    : float = 0.2
    random_state : int   = 42

    # ── Tiền xử lý ─────────────────────────────────────────────────────────
    resize_max : int  = 512
    use_clahe  : bool = True

    # ── Dense SIFT ─────────────────────────────────────────────────────────
    dsift_step         : int   = 8
    dsift_sizes        : tuple = (6, 8, 10, 12)
    max_desc_per_image : int   = 1500

    # ── PCA ────────────────────────────────────────────────────────────────
    # None = chưa chọn, sẽ set sau khi xem biểu đồ elbow (Cell 10)
    pca_dim      : int  = 40   # ← đặt sau khi xem Cell 10
    pca_whiten   : bool = True
    pca_max_desc : int  = 200_000
    pca_per_image: int  = 1000

    # ── SP-VLAD ────────────────────────────────────────────────────────────
    # None = chưa chọn, sẽ set sau khi xem biểu đồ elbow (Cell 13)
    vlad_k         : int   = 32  # ← đặt sau khi xem Cell 13
    vlad_max_desc  : int   = 400_000
    vlad_per_image : int   = 1500
    vlad_batch_size: int   = 16384
    spm_levels     : tuple = (0, 1, 2)
    intra_norm     : bool  = True
    power_norm     : bool  = True

    # ── IFV / Fisher Vector ────────────────────────────────────────────────
    ifv_k          : int   = 16
    ifv_max_desc   : int   = 300_000
    ifv_per_image  : int   = 1500
    ifv_reg_covar  : float = 1e-4
    ifv_power_alpha: float = 0.5

    # ── Flip augmentation ──────────────────────────────────────────────────
    flip_train     : bool = True
    flip_train_mode: str  = "avg"
    flip_test      : bool = True

    # ── HOG ────────────────────────────────────────────────────────────────
    use_hog  : bool = True
    hog_win  : int  = 128
    hog_cell : int  = 8
    hog_block: int  = 16
    hog_bins : int  = 9

    # ── LBP ────────────────────────────────────────────────────────────────
    use_lbp : bool = True
    lbp_grid: int  = 2

    # ── GLCM Haralick ──────────────────────────────────────────────────────
    use_glcm      : bool  = True
    glcm_levels   : int   = 32
    glcm_distances: tuple = (1, 3)
    glcm_grid     : int   = 2

    # ── GIST ───────────────────────────────────────────────────────────────
    use_gist         : bool = True
    gist_orientations: int  = 8
    gist_scales      : int  = 4
    gist_spatial     : int  = 4
    gist_img_size    : int  = 128

    # ── SVM & Ensemble ─────────────────────────────────────────────────────
    grid_cv_splits: int   = 5
    grid_n_jobs   : int   = 2
    oof_splits    : int   = 5
    nm_maxiter    : int   = 2000
    nm_xatol      : float = 1e-4


CFG = Config()
print('✅ Config OK')
print(f'   data_dir  = {CFG.data_dir}')
print(f'   ckpt_dir  = {CFG.ckpt_dir}')
print(f'   pca_dim   = {CFG.pca_dim}  ← sẽ chọn sau khi xem biểu đồ Cell 10')
print(f'   vlad_k    = {CFG.vlad_k}   ← sẽ chọn sau khi xem biểu đồ Cell 13')

---
## Cell 4 — Hàm checkpoint (local ↔ Drive sync)

In [ ]:
def _setup_dirs():
    CFG.local_ckpt.mkdir(parents=True, exist_ok=True)
    CFG.ckpt_dir.mkdir(parents=True, exist_ok=True)

def _local(name):  return CFG.local_ckpt / name
def _drive(name):  return CFG.ckpt_dir   / name

def exists(name):
    """Kiểm tra file tồn tại ở local hoặc Drive."""
    return _local(name).exists() or _drive(name).exists()

def _ensure_local(name):
    """Copy file từ Drive về local nếu chưa có local."""
    if not _local(name).exists() and _drive(name).exists():
        shutil.copy2(_drive(name), _local(name))
        print(f'  📥 Drive→local: {name}')

def _sync_to_drive(name):
    """Copy file từ local lên Drive (backup)."""
    if _local(name).exists():
        shutil.copy2(_local(name), _drive(name))
        print(f'  ☁️  local→Drive: {name}  ({_drive(name).stat().st_size/1e6:.1f} MB)')

def save_pkl(obj, name):
    _setup_dirs()
    with open(_local(name), 'wb') as f: pickle.dump(obj, f, protocol=5)
    _sync_to_drive(name)

def load_pkl(name):
    _setup_dirs()
    _ensure_local(name)
    with open(_local(name), 'rb') as f: obj = pickle.load(f)
    print(f'  📂 {name}')
    return obj

def save_npz(name, **arrays):
    _setup_dirs()
    np.savez_compressed(str(_local(name)), **arrays)
    _sync_to_drive(name)

def load_npz(name):
    _setup_dirs()
    _ensure_local(name)
    data = np.load(str(_local(name)), allow_pickle=True)
    out  = {k: data[k] for k in data.files}
    print(f'  📂 {name}  keys={list(out.keys())}')
    return out

def save_fig(fig, name):
    """Lưu figure cả local lẫn Drive."""
    _setup_dirs()
    fig.savefig(_local(name), dpi=150, bbox_inches='tight')
    shutil.copy2(_local(name), _drive(name))
    print(f'  🖼️  {name} → Drive')

def checkpoint_status():
    files = ['labels.npz','pca_raw_descriptors.npz','pca.pkl',
             'kmeans.pkl','gmm.pkl','features_train.npz',
             'features_test.npz','scalers_train.pkl',
             'clfs.pkl','weights.pkl','results.pkl']
    print('\n── Checkpoint status ────────────────────────────────────')
    for f in files:
        lo = _local(f).exists()
        dr = _drive(f).exists()
        tag = ('✅' if lo else '⬜') + '/' + ('☁️ ' if dr else '⬜')
        sz  = f"{_drive(f).stat().st_size/1e6:.1f}MB" if dr else '-'
        print(f'  {tag}  {f:<32s}  {sz}')
    print('   (✅=local  ☁️=Drive)')
    print('────────────────────────────────────────────────────────\n')

_setup_dirs()
print('✅ Checkpoint utilities OK')
print(f'   Local cache : {CFG.local_ckpt}')
print(f'   Drive backup: {CFG.ckpt_dir}')

---
## Cell 5 — Hàm xử lý ảnh & singletons

In [ ]:
_CLAHE = None
_HOG   = None
_GABOR_KERNELS = None

def get_clahe():
    global _CLAHE
    if _CLAHE is None:
        _CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return _CLAHE

def get_hog(cfg):
    global _HOG
    if _HOG is None:
        _HOG = cv2.HOGDescriptor(
            (cfg.hog_win,cfg.hog_win),(cfg.hog_block,cfg.hog_block),
            (cfg.hog_cell,cfg.hog_cell),(cfg.hog_cell,cfg.hog_cell),cfg.hog_bins)
    return _HOG

def get_gabor_kernels(cfg):
    global _GABOR_KERNELS
    if _GABOR_KERNELS is None:
        _GABOR_KERNELS = []
        for scale in range(cfg.gist_scales):
            sigma = 2**scale * 2.0
            for orient in range(cfg.gist_orientations):
                theta = orient * np.pi / cfg.gist_orientations
                ksize = int(6*sigma+1)|1
                _GABOR_KERNELS.append(
                    cv2.getGaborKernel((ksize,ksize),sigma,theta,sigma*2,0.5,0))
    return _GABOR_KERNELS

def _resize(img, max_side):
    if max_side is None: return img
    h,w = img.shape[:2]; m = max(h,w)
    if m <= max_side: return img
    s = max_side/m
    return cv2.resize(img,(int(w*s),int(h*s)),interpolation=cv2.INTER_AREA)

def read_bgr_to_gray(path, cfg):
    """Đọc 1 lần BGR → trả (bgr, gray) đã resize + CLAHE."""
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None: return None, None
    bgr  = _resize(bgr, cfg.resize_max)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    if cfg.use_clahe: gray = get_clahe().apply(gray)
    return bgr, gray

def read_gray(path, cfg):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img = _resize(img, cfg.resize_max)
    if cfg.use_clahe: img = get_clahe().apply(img)
    return img

print('✅ Singletons & IO OK')

---
## Cell 6 — Hàm Dense SIFT + RootSIFT & tất cả đặc trưng

In [ ]:
# ── RootSIFT ──────────────────────────────────────────────────────────────
def rootsift(des, eps=1e-8):
    des = des.astype(np.float32)
    des /= des.sum(1,keepdims=True)+eps
    return normalize(np.sqrt(des), norm='l2').astype(np.float32)

def dense_kps(h,w,step,size):
    return [cv2.KeyPoint(float(x),float(y),float(size))
            for y in np.arange(step//2,h,step,dtype=np.float32)
            for x in np.arange(step//2,w,step,dtype=np.float32)]

def extract_dense_sift(img, sift, cfg, rng):
    h,w = img.shape[:2]; all_k,all_d = [],[]
    for sz in cfg.dsift_sizes:
        kps = dense_kps(h,w,cfg.dsift_step,sz)
        if not kps: continue
        kps,des = sift.compute(img,kps)
        if des is None or len(des)==0: continue
        all_k.append(kps); all_d.append(des)
    if not all_d: return None,None,(h,w)
    kps_flat = [k for g in all_k for k in g]
    des = np.vstack(all_d).astype(np.float32)
    if len(des)>cfg.max_desc_per_image:
        idx = rng.choice(len(des),cfg.max_desc_per_image,replace=False)
        pts = np.array([kps_flat[i].pt for i in idx],dtype=np.float32); des=des[idx]
    else:
        pts = np.array([k.pt for k in kps_flat],dtype=np.float32)
    return pts, rootsift(des), (h,w)

def extract_one(path, sift, cfg, rng, flip=False):
    img = read_gray(path, cfg)
    if img is None: return None,None,None
    if flip: img = cv2.flip(img,1)
    return extract_dense_sift(img, sift, cfg, rng)

def sample_des(des, take, rng):
    if des is None or len(des)==0 or take<=0: return None
    if len(des)<=take: return des
    return des[rng.choice(len(des),take,replace=False)]

# ── SP-VLAD (vectorized) ──────────────────────────────────────────────────
@lru_cache(maxsize=1)
def _spm_regions(levels):
    regs=[]
    for l in levels:
        b=2**l
        for ry in range(b):
            for rx in range(b): regs.append((rx,ry,b))
    return regs

def _vlad_block(des, centers, K, D):
    if des is None or len(des)==0: return np.zeros((K,D),dtype=np.float32)
    words = np.argmin(np.sum((des[:,None,:]-centers[None,:,:])**2,2),1)
    V = np.zeros((K,D),dtype=np.float32)
    np.add.at(V, words, des-centers[words])
    return normalize(V, norm='l2').astype(np.float32)

def _region_mask(pts, shape, rx, ry, bins):
    h,w=shape
    return ((pts[:,0]>=rx/bins*w)&(pts[:,0]<(rx+1)/bins*w)&
            (pts[:,1]>=ry/bins*h)&(pts[:,1]<(ry+1)/bins*h))

def spvlad_feature(pts, des, shape, kmeans, cfg):
    regs=_spm_regions(cfg.spm_levels)
    centers=kmeans.cluster_centers_.astype(np.float32)
    K,D=cfg.vlad_k,cfg.pca_dim
    if des is None or pts is None: return np.zeros(len(regs)*K*D,dtype=np.float32)
    blocks=[]
    for rx,ry,bins in regs:
        m=_region_mask(pts,shape,rx,ry,bins)
        blocks.append(_vlad_block(des[m] if np.any(m) else None,centers,K,D).reshape(-1))
    feat=np.concatenate(blocks).astype(np.float32)
    if cfg.power_norm: feat=np.sign(feat)*np.sqrt(np.abs(feat)+1e-12)
    return normalize(feat.reshape(1,-1),norm='l2').ravel().astype(np.float32)

def build_vlad(paths, sift, pca, kmeans, cfg, flip=False):
    rng=np.random.default_rng(cfg.random_state)
    regs=_spm_regions(cfg.spm_levels)
    X=np.zeros((len(paths),len(regs)*cfg.vlad_k*cfg.pca_dim),dtype=np.float32)
    for i,p in enumerate(tqdm(paths,desc=f"VLAD{'↔' if flip else ''}",unit='img')):
        pts,des,shape=extract_one(p,sift,cfg,rng,flip)
        if des is not None and len(des)>0: des=pca.transform(des).astype(np.float32)
        X[i]=spvlad_feature(pts,des,shape,kmeans,cfg)
    return X

# ── IFV / Fisher Vector ───────────────────────────────────────────────────
def fisher_vector(des, gmm, cfg, eps=1e-9):
    if des is None or len(des)==0: return np.zeros(2*cfg.ifv_k*cfg.pca_dim,dtype=np.float32)
    w=gmm.weights_.astype(np.float32); mu=gmm.means_.astype(np.float32)
    sigma=np.maximum(gmm.covariances_.astype(np.float32),eps)
    q=gmm.predict_proba(des.astype(np.float64)).astype(np.float32)
    N,qT=des.shape[0],q.T; x=des.astype(np.float32); qx=qT@x
    s0=np.maximum(q.sum(0),eps)
    u=(qx-s0[:,None]*mu)/(np.sqrt(sigma)*(N*np.sqrt(w+eps)[:,None]+eps))
    qx2=qT@(x*x)
    v=((qx2-2*mu*qx+s0[:,None]*(mu*mu))/sigma-s0[:,None])/(N*np.sqrt(2*w+eps)[:,None]+eps)
    fv=np.concatenate([u.reshape(-1),v.reshape(-1)]).astype(np.float32)
    fv=np.sign(fv)*(np.abs(fv)**cfg.ifv_power_alpha)
    return (fv/(np.linalg.norm(fv)+eps)).astype(np.float32)

def build_ifv(paths, sift, pca, gmm, cfg, flip=False):
    rng=np.random.default_rng(cfg.random_state)
    X=np.zeros((len(paths),2*cfg.ifv_k*cfg.pca_dim),dtype=np.float32)
    for i,p in enumerate(tqdm(paths,desc=f"IFV{'↔' if flip else ''}",unit='img')):
        _,des,_=extract_one(p,sift,cfg,rng,flip)
        if des is not None and len(des)>0: des=pca.transform(des).astype(np.float32)
        X[i]=fisher_vector(des,gmm,cfg)
    return X

# ── HOG ──────────────────────────────────────────────────────────────────
def hog_feature(gray, cfg):
    img=cv2.resize(gray,(cfg.hog_win,cfg.hog_win),interpolation=cv2.INTER_AREA)
    feat=get_hog(cfg).compute(img).reshape(-1).astype(np.float32)
    return feat/(np.linalg.norm(feat)+1e-12)

# ── LBP ──────────────────────────────────────────────────────────────────
def _lbp_codes(gray):
    g=gray.astype(np.uint8); c=g[1:-1,1:-1]; codes=np.zeros_like(c,dtype=np.uint8)
    nbs=[g[0:-2,0:-2],g[0:-2,1:-1],g[0:-2,2:],g[1:-1,2:],
         g[2:,2:],g[2:,1:-1],g[2:,0:-2],g[1:-1,0:-2]]
    for i,nb in enumerate(nbs): codes|=((nb>=c)<<(7-i)).astype(np.uint8)
    return codes

def lbp_feature(gray, cfg):
    img=cv2.resize(gray,(cfg.hog_win,cfg.hog_win),interpolation=cv2.INTER_AREA)
    codes=_lbp_codes(img); h,w,g=codes.shape[0],codes.shape[1],cfg.lbp_grid
    feats=[]
    for iy in range(g):
        for ix in range(g):
            p=codes[int(iy*h/g):int((iy+1)*h/g),int(ix*w/g):int((ix+1)*w/g)].ravel()
            hist=np.bincount(p,minlength=256).astype(np.float32)
            feats.append(hist/(hist.sum()+1e-12))
    return np.concatenate(feats).astype(np.float32)

# ── GLCM Haralick (thay HSV) ──────────────────────────────────────────────
def glcm_feature(gray, cfg):
    img=cv2.resize(gray,(cfg.hog_win,cfg.hog_win),interpolation=cv2.INTER_AREA)
    img_q=np.clip(img//(256//cfg.glcm_levels),0,cfg.glcm_levels-1).astype(np.uint8)
    angles=[0,np.pi/4,np.pi/2,3*np.pi/4]
    props=['contrast','dissimilarity','homogeneity','energy','correlation']
    h,w,g=img_q.shape[0],img_q.shape[1],cfg.glcm_grid
    feats=[]
    for iy in range(g):
        for ix in range(g):
            patch=img_q[int(iy*h/g):int((iy+1)*h/g),int(ix*w/g):int((ix+1)*w/g)]
            glcm=graycomatrix(patch,distances=list(cfg.glcm_distances),
                              angles=angles,levels=cfg.glcm_levels,symmetric=True,normed=True)
            for prop in props:
                v=graycoprops(glcm,prop)
                feats.extend([v.mean(),v.std()])
    fv=np.array(feats,dtype=np.float32)
    return fv/(np.linalg.norm(fv)+1e-12)

# ── GIST (Gabor spatial envelope) ─────────────────────────────────────────
def gist_feature(gray, cfg):
    img=cv2.resize(gray,(cfg.gist_img_size,cfg.gist_img_size),
                   interpolation=cv2.INTER_AREA).astype(np.float32)/255.0
    h,w,g=img.shape[0],img.shape[1],cfg.gist_spatial
    kernels=get_gabor_kernels(cfg); feats=[]
    for kernel in kernels:
        resp=np.abs(cv2.filter2D(img,cv2.CV_32F,kernel))
        for gy in range(g):
            for gx in range(g):
                feats.append(resp[int(gy*h/g):int((gy+1)*h/g),
                                  int(gx*w/g):int((gx+1)*w/g)].mean())
    fv=np.array(feats,dtype=np.float32)
    fv=fv/(fv.sum()+1e-12)
    fv=np.sign(fv)*np.sqrt(np.abs(fv)+1e-12)
    return fv/(np.linalg.norm(fv)+1e-12)

# ── Build extra (HOG+LBP+GLCM+GIST) — đọc ảnh 1 lần ─────────────────────
def build_extra(paths, cfg, flip=False):
    get_gabor_kernels(cfg)  # pre-build
    lists={k:[] for k in ['hog','lbp','glcm','gist']}
    for p in tqdm(paths,desc=f"Extra{'↔' if flip else ''}",unit='img'):
        _,gray=read_bgr_to_gray(p,cfg)
        if gray is None:
            for k in lists: lists[k].append(None)
            continue
        if flip: gray=cv2.flip(gray,1)
        if cfg.use_hog:  lists['hog'].append(hog_feature(gray,cfg))
        if cfg.use_lbp:  lists['lbp'].append(lbp_feature(gray,cfg))
        if cfg.use_glcm: lists['glcm'].append(glcm_feature(gray,cfg))
        if cfg.use_gist: lists['gist'].append(gist_feature(gray,cfg))
    out={}
    for k,lst in lists.items():
        valid=[x for x in lst if x is not None]
        if not valid: continue
        dim=len(valid[0])
        out[k]=np.vstack([x if x is not None else np.zeros(dim,np.float32) for x in lst]).astype(np.float32)
    return out

print('✅ Tất cả hàm đặc trưng sẵn sàng')

---
## Cell 7 — Mount Google Drive  🔑

In [ ]:
from google.colab import drive
import shutil

# Unmount Google Drive first to ensure a clean state
if Path('/content/drive').is_mount():
    try:
        drive.flush_and_unmount('/content/drive')
        print("Drive unmounted successfully.")
    except Exception as e:
        print(f"Error during unmount: {e}")

# Explicitly ensure the mount point is empty before attempting to mount
mount_point = Path('/content/drive')
if mount_point.is_dir() and list(mount_point.iterdir()):
    print(f"Mount point {mount_point} is not empty. Clearing contents...")
    try:
        shutil.rmtree(mount_point)
        mount_point.mkdir(parents=True, exist_ok=True) # Recreate empty directory
        print("Mount point cleared and recreated.")
    except Exception as e:
        print(f"Error clearing mount point: {e}")

drive.mount('/content/drive', force_remount=True)

# Kiểm tra dữ liệu tồn tại
if not CFG.data_dir.exists():
    raise FileNotFoundError(
        f'Không tìm thấy thư mục dữ liệu: {CFG.data_dir}\n'
        f'Hãy kiểm tra đường dẫn trong Config (Cell 3).'
    )

classes_found = sorted([p.name for p in CFG.data_dir.iterdir() if p.is_dir()])
print(f'✅ Drive mounted OK')
print(f'   data_dir : {CFG.data_dir}')
print(f'   ckpt_dir : {CFG.ckpt_dir}  (sẽ tạo nếu chưa có)')
print(f'   Tìm thấy {len(classes_found)} lớp: {classes_found}')
CFG.ckpt_dir.mkdir(parents=True, exist_ok=True)
checkpoint_status()

In [ ]:
# Cell huấn luyện cho đặc trưng VLAD với SVM (RBF kernel)
print(f'\n─── VLAD (dim={X_dict_tr["vlad"].shape[1]}) ───')
candidate_models = [
    (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']})
]

clf, best_params, best_cv_score = train_best_classifier_for_feature(
    X_dict_tr['vlad'], ytr, CFG, 'vlad', candidate_models
)
clfs['vlad'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
save_pkl(clfs, 'clfs.pkl')

print(f"  VLAD Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")

In [ ]:
# Cell huấn luyện cho đặc trưng IFV với SVM (RBF kernel)
print(f'\n─── IFV (dim={X_dict_tr["ifv"].shape[1]}) ───')
candidate_models = [
    (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']})
]

clf, best_params, best_cv_score = train_best_classifier_for_feature(
    X_dict_tr['ifv'], ytr, CFG, 'ifv', candidate_models
)
clfs['ifv'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
save_pkl(clfs, 'clfs.pkl')

print(f"  IFV Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")

In [ ]:
# Cell huấn luyện cho đặc trưng HOG với SVM (RBF kernel)
if 'hog' in X_dict_tr:
    print(f'\n─── HOG (dim={X_dict_tr["hog"].shape[1]}) ───')
    candidate_models = [
        (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']})
    ]

    clf, best_params, best_cv_score = train_best_classifier_for_feature(
        X_dict_tr['hog'], ytr, CFG, 'hog', candidate_models
    )
    clfs['hog'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
    save_pkl(clfs, 'clfs.pkl')
    print(f"  HOG Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")
else:
    print('\n─── HOG: Not used (cfg.use_hog is False) ───')
    # If HOG is not used, ensure it's not in clfs if it was previously loaded from checkpoint
    if 'hog' in clfs: del clfs['hog']; save_pkl(clfs, 'clfs.pkl')

In [ ]:
# Cell huấn luyện cho đặc trưng LBP với SVM (RBF kernel)
if 'lbp' in X_dict_tr:
    print(f'\n─── LBP (dim={X_dict_tr["lbp"].shape[1]}) ───')
    candidate_models = [
        (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']})
    ]

    clf, best_params, best_cv_score = train_best_classifier_for_feature(
        X_dict_tr['lbp'], ytr, CFG, 'lbp', candidate_models
    )
    clfs['lbp'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
    save_pkl(clfs, 'clfs.pkl')
    print(f"  LBP Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")
else:
    print('\n─── LBP: Not used (cfg.use_lbp is False) ───')
    if 'lbp' in clfs: del clfs['lbp']; save_pkl(clfs, 'clfs.pkl')

In [ ]:
# Cell huấn luyện cho đặc trưng GLCM Haralick với SVM (RBF kernel)
if 'glcm' in X_dict_tr:
    print(f'\n─── GLCM Haralick (dim={X_dict_tr["glcm"].shape[1]}) ───')
    candidate_models = [
        (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']})
    ]

    clf, best_params, best_cv_score = train_best_classifier_for_feature(
        X_dict_tr['glcm'], ytr, CFG, 'glcm', candidate_models
    )
    clfs['glcm'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
    save_pkl(clfs, 'clfs.pkl')
    print(f"  GLCM Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")
else:
    print('\n─── GLCM: Not used (cfg.use_glcm is False) ───')
    if 'glcm' in clfs: del clfs['glcm']; save_pkl(clfs, 'clfs.pkl')

In [ ]:
# Cell huấn luyện cho đặc trưng GIST với SVM (RBF kernel)
if 'gist' in X_dict_tr:
    print(f'\n─── GIST (dim={X_dict_tr["gist"].shape[1]}) ───')
    candidate_models = [
        (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']})
    ]

    clf, best_params, best_cv_score = train_best_classifier_for_feature(
        X_dict_tr['gist'], ytr, CFG, 'gist', candidate_models
    )
    clfs['gist'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
    save_pkl(clfs, 'clfs.pkl')
    print(f"  GIST Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")
else:
    print('\n─── GIST: Not used (cfg.use_gist is False) ───')
    if 'gist' in clfs: del clfs['gist']; save_pkl(clfs, 'clfs.pkl')

---
## Cell 8 — Load dataset & train/test split

In [ ]:
sift = cv2.SIFT_create()

if exists('labels.npz'):
    d=load_npz('labels.npz')
    X_train_p=d['X_train_p']; X_test_p=d['X_test_p']
    y_train=d['y_train'];     y_test=d['y_test']
    classes=list(d['classes'])
else:
    classes=sorted([p.name for p in CFG.data_dir.iterdir() if p.is_dir()])
    if not classes: raise FileNotFoundError(f'Không tìm thấy lớp trong {CFG.data_dir}')
    c2i={c:i for i,c in enumerate(classes)}
    paths,labels=[],[]
    for c in classes:
        for ext in CFG.img_exts:
            for p in sorted(glob.glob(str(CFG.data_dir/c/ext))):
                paths.append(p); labels.append(c2i[c])
    if not paths: raise FileNotFoundError('Không tìm thấy ảnh. Kiểm tra DATA_DIR và img_exts.')
    paths=np.array(paths); labels=np.array(labels)
    X_train_p,X_test_p,y_train,y_test=train_test_split(
        paths,labels,test_size=CFG.test_size,
        random_state=CFG.random_state,stratify=labels)
    save_npz('labels.npz',X_train_p=X_train_p,X_test_p=X_test_p,
             y_train=y_train,y_test=y_test,classes=np.array(classes))

print(f'\n📦 {len(classes)} classes | train={len(X_train_p)} | test={len(X_test_p)}')
print(f'   Ảnh/lớp (train) ≈ {len(X_train_p)//len(classes)}')
print(f'   Classes: {classes}')

---
## Cell 9 — Sample raw descriptors (cần cho biểu đồ PCA & KMeans)

In [ ]:
# Raw descriptors dùng để:
#   1) Vẽ biểu đồ PCA explained variance (Cell 10)
#   2) Vẽ biểu đồ KMeans inertia elbow (Cell 13)
# Chỉ sample 1 lần và lưu lại Drive.

if exists('pca_raw_descriptors.npz'):
    d=load_npz('pca_raw_descriptors.npz')
    X_raw=d['X_raw']
    print(f'✅ Raw descriptors loaded: shape={X_raw.shape}')
else:
    rng=np.random.default_rng(CFG.random_state)
    pool,total=[],0
    for p in tqdm(X_train_p,desc='Sampling raw descriptors',unit='img'):
        _,des,_=extract_one(p,sift,CFG,rng)
        if des is None: continue
        take=min(CFG.pca_per_image,CFG.pca_max_desc-total)
        s=sample_des(des,take,rng)
        if s is None: continue
        pool.append(s); total+=len(s)
        if total>=CFG.pca_max_desc: break
    if total==0: raise RuntimeError('Không sample được descriptor nào.')
    X_raw=np.vstack(pool).astype(np.float32)
    del pool; gc.collect()
    save_npz('pca_raw_descriptors.npz',X_raw=X_raw)
    print(f'✅ Raw descriptors: shape={X_raw.shape}')

---
## Cell 10 — 📊 Biểu đồ PCA Explained Variance (chọn `pca_dim`)

> Chạy cell này, xem biểu đồ, rồi **đặt `CFG.pca_dim`** ở Cell 11.

In [ ]:
# Fit PCA đầy đủ (không giảm chiều) để xem phân bố variance
pca_full = PCA(svd_solver='randomized', random_state=CFG.random_state)
pca_full.fit(X_raw)

ev     = pca_full.explained_variance_ratio_
ev_cum = np.cumsum(ev)
n_comp = np.arange(1, len(ev)+1)

# ── Tìm điểm elbow tự động (kneedle approximation) ───────────────────────
# Đường thẳng từ điểm đầu đến cuối, tìm điểm xa nhất
x_norm = (n_comp - n_comp[0]) / (n_comp[-1] - n_comp[0])
y_norm = (ev_cum - ev_cum[0]) / (ev_cum[-1] - ev_cum[0])
dist   = np.abs(y_norm - x_norm)
elbow_idx = int(np.argmax(dist))  # index trong toàn bộ mảng
elbow_n   = n_comp[elbow_idx]

# Các ngưỡng thông dụng
thresholds = [0.80, 0.85, 0.90, 0.95]
thresh_n   = [int(np.searchsorted(ev_cum, t))+1 for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Panel trái: cumulative explained variance ──────────────────────────────
ax = axes[0]
ax.plot(n_comp[:128], ev_cum[:128], color='#378ADD', linewidth=2)
ax.axvline(elbow_n, color='#E24B4A', linestyle='--', linewidth=1.5,
           label=f'Elbow ≈ {elbow_n}d')
colors_t = ['#888780','#5F5E5A','#444441','#2C2C2A']
for t, n, c in zip(thresholds, thresh_n, colors_t):
    if n <= 128:
        ax.axvline(n, color=c, linestyle=':', linewidth=1,
                   label=f'{int(t*100)}% → {n}d')
ax.set_xlabel('Số components'); ax.set_ylabel('Cumulative explained variance')
ax.set_title('PCA — Cumulative Variance (first 128 dims)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_xlim(1, 128); ax.set_ylim(0, 1)

# ── Panel phải: marginal (từng component) ─────────────────────────────────
ax2 = axes[1]
ax2.bar(n_comp[:64], ev[:64]*100, color='#B5D4F4', edgecolor='#378ADD',
        linewidth=0.3, width=0.8)
ax2.axvline(elbow_n, color='#E24B4A', linestyle='--', linewidth=1.5,
            label=f'Elbow ≈ {elbow_n}d')
ax2.set_xlabel('Component'); ax2.set_ylabel('Explained variance (%)')
ax2.set_title('PCA — Per-component variance (first 64)')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
save_fig(fig, 'pca_elbow.png')
plt.show()

print('\n── Gợi ý chọn pca_dim ──────────────────────────────')
print(f'  Elbow tự động          : {elbow_n} components')
for t, n in zip(thresholds, thresh_n):
    at = ev_cum[n-1] if n <= len(ev_cum) else 1.0
    print(f'  Ngưỡng {int(t*100)}% variance    : {n:3d} components  (actual={at:.3f})')
print(f'\n  Với 4485 ảnh, an toàn khi pca_dim ≤ {len(X_train_p)//10} (rule: n_samples/10)')
print('\n→ Đặt CFG.pca_dim ở Cell 11 rồi chạy tiếp.')

---
## Cell 11 — ✏️  Chọn `pca_dim` và fit PCA

In [ ]:
# ════════════════════════════════════════════════════════
#  ← CHỈNH TẠI ĐÂY sau khi xem biểu đồ Cell 10
CFG.pca_dim = 40
# ════════════════════════════════════════════════════════

n_train = len(X_train_p)
ratio   = n_train / CFG.pca_dim
status  = '✅ An toàn' if ratio >= 10 else ('⚠️  Chấp nhận được' if ratio >= 5 else '❌ Quá cao, giảm pca_dim')
print(f'pca_dim = {CFG.pca_dim}')
print(f'Tỉ lệ mẫu/chiều = {n_train}/{CFG.pca_dim} = {ratio:.1f}×  {status}')
print(f'Variance giải thích: {pca_full.explained_variance_ratio_[:CFG.pca_dim].sum():.3f}')

if exists('pca.pkl'):
    pca = load_pkl('pca.pkl')
    if pca.n_components_ != CFG.pca_dim:
        print(f'⚠️  pca.pkl có {pca.n_components_}d khác {CFG.pca_dim}d → refit')
        os.remove(_local('pca.pkl'))
        if _drive('pca.pkl').exists(): os.remove(_drive('pca.pkl'))
        pca = None
else:
    pca = None

if pca is None:
    pca = PCA(n_components=CFG.pca_dim, whiten=CFG.pca_whiten,
              random_state=CFG.random_state, svd_solver='randomized')
    pca.fit(X_raw)
    save_pkl(pca, 'pca.pkl')

print(f'\n✅ PCA {pca.n_components_}d  |  var_explained={pca.explained_variance_ratio_.sum():.4f}')

---
## Cell 12 — Sample PCA-projected descriptors (cần cho KMeans elbow)

In [ ]:
# Project raw → PCA space để vẽ KMeans elbow
# (dùng lại X_raw đã sample ở Cell 9)
rng = np.random.default_rng(CFG.random_state)
take = min(CFG.vlad_max_desc, len(X_raw))
idx  = rng.choice(len(X_raw), take, replace=False)
X_pca_proj = pca.transform(X_raw[idx]).astype(np.float32)
print(f'✅ X_pca_proj shape = {X_pca_proj.shape}  (dùng để vẽ KMeans elbow)')

---
## Cell 13 — 📊 Biểu đồ KMeans Inertia Elbow (chọn `vlad_k`)

> Chạy cell này, xem biểu đồ, rồi **đặt `CFG.vlad_k`** ở Cell 14.
>
> ⏱️  Cell này mất khoảng 2–5 phút tùy số K cần thử.

In [ ]:
# Danh sách K cần thử — điều chỉnh nếu cần
K_candidates = [8, 16, 24, 32, 48, 64, 96, 128]

print('Đang tính inertia cho từng K... (có thể mất vài phút)')
inertias = []
for k in tqdm(K_candidates, desc='KMeans elbow'):
    km = MiniBatchKMeans(n_clusters=k, random_state=CFG.random_state,
                         batch_size=CFG.vlad_batch_size, n_init=3,
                         reassignment_ratio=0.01, max_iter=100)
    km.fit(X_pca_proj)
    inertias.append(km.inertia_)
    print(f'  K={k:3d}  inertia={km.inertia_:.2f}')

# ── Tìm elbow tự động ─────────────────────────────────────────────────────
K_arr = np.array(K_candidates, dtype=float)
I_arr = np.array(inertias, dtype=float)
x_n   = (K_arr - K_arr[0])/(K_arr[-1]-K_arr[0])
y_n   = (I_arr - I_arr[0])/(I_arr[-1]-I_arr[0])
elbow_k_idx = int(np.argmax(np.abs(y_n - x_n)))
elbow_k     = K_candidates[elbow_k_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_candidates, inertias, 'o-', color='#378ADD', linewidth=2, markersize=6)
ax.axvline(elbow_k, color='#E24B4A', linestyle='--', linewidth=1.5,
           label=f'Elbow ≈ K={elbow_k}')
ax.set_xlabel('K (số cluster)'); ax.set_ylabel('Inertia')
ax.set_title('KMeans Inertia Elbow — chọn vlad_k')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
# Chú thích từng điểm
for k, ine in zip(K_candidates, inertias):
    ax.annotate(str(k), (k, ine), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8, color='#5F5E5A')
plt.tight_layout()
save_fig(fig, 'kmeans_elbow.png')
plt.show()

print(f'\n→ Elbow tự động gợi ý: K = {elbow_k}')
n_train = len(X_train_p)
print('\n── Tỉ lệ mẫu/chiều VLAD với từng K ──')
regs = _spm_regions(CFG.spm_levels)
for k in K_candidates:
    dim   = len(regs)*k*CFG.pca_dim
    ratio = n_train/dim
    flag  = '✅' if ratio>=0.1 else ('⚠️' if ratio>=0.05 else '❌')
    print(f'  K={k:3d}  VLAD dim={dim:6d}  ratio={ratio:.3f}×  {flag}')
print('\n→ Đặt CFG.vlad_k ở Cell 14 rồi chạy tiếp.')

---
## Cell 14 — ✏️  Chọn `vlad_k` và fit KMeans

In [ ]:
# ════════════════════════════════════════════════════════
#  ← CHỈNH TẠI ĐÂY sau khi xem biểu đồ Cell 13
CFG.vlad_k = 32
# ════════════════════════════════════════════════════════

regs  = _spm_regions(CFG.spm_levels)
vlad_dim = len(regs)*CFG.vlad_k*CFG.pca_dim
ratio = len(X_train_p)/vlad_dim
status = '✅ An toàn' if ratio>=0.1 else ('⚠️  Chấp nhận được' if ratio>=0.05 else '❌ Quá cao')
print(f'vlad_k = {CFG.vlad_k}')
print(f'VLAD dim = {len(regs)} × {CFG.vlad_k} × {CFG.pca_dim} = {vlad_dim}')
print(f'Tỉ lệ mẫu/chiều = {len(X_train_p)}/{vlad_dim} = {ratio:.3f}×  {status}')

# ── Fit KMeans ────────────────────────────────────────────────────────────
if exists('kmeans.pkl'):
    kmeans = load_pkl('kmeans.pkl')
    if kmeans.n_clusters != CFG.vlad_k:
        print(f'⚠️  kmeans.pkl có K={kmeans.n_clusters} khác {CFG.vlad_k} → refit')
        kmeans = None
else:
    kmeans = None

if kmeans is None:
    rng=np.random.default_rng(CFG.random_state)
    kmeans=MiniBatchKMeans(n_clusters=CFG.vlad_k,random_state=CFG.random_state,
                           batch_size=CFG.vlad_batch_size,n_init=3,reassignment_ratio=0.01)
    buf,buf_n,seen,fitted=[],0,0,False
    for p in tqdm(X_train_p,desc='KMeans fit',unit='img'):
        _,des,_=extract_one(p,sift,CFG,rng)
        if des is None: continue
        des_p=pca.transform(des).astype(np.float32)
        take=min(CFG.vlad_per_image,CFG.vlad_max_desc-seen)
        s=sample_des(des_p,take,rng)
        if s is None: continue
        buf.append(s); buf_n+=len(s); seen+=len(s)
        if buf_n>=CFG.vlad_batch_size:
            B=np.vstack(buf).astype(np.float32); buf.clear(); buf_n=0
            if (not fitted) and len(B)<CFG.vlad_k: continue
            kmeans.partial_fit(B); fitted=True; del B; gc.collect()
        if seen>=CFG.vlad_max_desc: break
    if buf_n>0:
        B=np.vstack(buf).astype(np.float32)
        if len(B)>=CFG.vlad_k or fitted: kmeans.partial_fit(B); fitted=True
    if not fitted: raise RuntimeError('KMeans chưa fit.')
    save_pkl(kmeans,'kmeans.pkl')

print(f'\n✅ KMeans K={kmeans.n_clusters}  |  inertia={kmeans.inertia_:.2f}')

---
## Cell 15 — Fit GMM (IFV)

In [ ]:
if exists('gmm.pkl'):
    gmm=load_pkl('gmm.pkl')
else:
    rng=np.random.default_rng(CFG.random_state)
    pool,total=[],0
    for p in tqdm(X_train_p,desc='GMM sampling',unit='img'):
        _,des,_=extract_one(p,sift,CFG,rng)
        if des is None: continue
        des_p=pca.transform(des).astype(np.float32)
        take=min(CFG.ifv_per_image,CFG.ifv_max_desc-total)
        s=sample_des(des_p,take,rng)
        if s is None: continue
        pool.append(s); total+=len(s)
        if total>=CFG.ifv_max_desc: break
    if total<CFG.ifv_k: raise RuntimeError('Không đủ descriptor cho GMM.')
    X_gmm=np.vstack(pool).astype(np.float64); del pool; gc.collect()
    gmm=GaussianMixture(n_components=CFG.ifv_k,covariance_type='diag',
                        reg_covar=CFG.ifv_reg_covar,max_iter=200,
                        random_state=CFG.random_state,init_params='kmeans')
    gmm.fit(X_gmm); del X_gmm; gc.collect()
    save_pkl(gmm,'gmm.pkl')

print(f'✅ GMM K={gmm.n_components}  |  converged={gmm.converged_}  |  lb={gmm.lower_bound_:.4f}')

---
## Cell 16 — Build đặc trưng TRAIN

In [ ]:
if exists('features_train.npz') and exists('scalers_train.pkl'):
    d=load_npz('features_train.npz')
    scalers=load_pkl('scalers_train.pkl')
    ytr=d['ytr']; X_dict_tr={k:d[k] for k in d if k!='ytr'}
else:
    Xv=build_vlad(X_train_p,sift,pca,kmeans,CFG,flip=False)
    Xi=build_ifv(X_train_p,sift,pca,gmm,CFG,flip=False)
    ytr=y_train.copy()
    if CFG.flip_train:
        Xv_f=build_vlad(X_train_p,sift,pca,kmeans,CFG,flip=True)
        Xi_f=build_ifv(X_train_p,sift,pca,gmm,CFG,flip=True)
        if CFG.flip_train_mode=='stack':
            Xv=np.vstack([Xv,Xv_f]).astype(np.float32)
            Xi=np.vstack([Xi,Xi_f]).astype(np.float32)
            ytr=np.hstack([y_train,y_train])
        else:
            Xv=normalize(Xv+Xv_f,norm='l2').astype(np.float32)
            Xi=normalize(Xi+Xi_f,norm='l2').astype(np.float32)
        del Xv_f,Xi_f; gc.collect()
    extra=build_extra(X_train_p,CFG,flip=False)
    if CFG.flip_train and extra:
        extra_f=build_extra(X_train_p,CFG,flip=True)
        for k in extra:
            extra[k]=(np.vstack([extra[k],extra_f[k]]).astype(np.float32)
                      if CFG.flip_train_mode=='stack'
                      else normalize(extra[k]+extra_f[k],norm='l2').astype(np.float32))
        del extra_f; gc.collect()
    scalers={}; X_dict_tr={'vlad':Xv,'ifv':Xi}
    for name,Xf in extra.items():
        sc=StandardScaler()
        X_dict_tr[name]=sc.fit_transform(Xf).astype(np.float32)
        scalers[name]=sc
    save_npz('features_train.npz',ytr=ytr,**X_dict_tr)
    save_pkl(scalers,'scalers_train.pkl')

print('\n📐 Chiều vector TRAIN:')
for k,v in X_dict_tr.items():
    print(f'   {k:>6s}: {v.shape}  (mẫu/chiều={len(ytr)/v.shape[1]:.2f}×)')

---
## Cell 17 — Build đặc trưng TEST

In [ ]:
if exists('features_test.npz'):
    d=load_npz('features_test.npz')
    X_dict_te={k:d[k] for k in d}
else:
    Xv_te=build_vlad(X_test_p,sift,pca,kmeans,CFG,flip=False)
    Xi_te=build_ifv(X_test_p,sift,pca,gmm,CFG,flip=False)
    if CFG.flip_test:
        Xv_f=build_vlad(X_test_p,sift,pca,kmeans,CFG,flip=True)
        Xi_f=build_ifv(X_test_p,sift,pca,gmm,CFG,flip=True)
        Xv_te=normalize(Xv_te+Xv_f,norm='l2').astype(np.float32)
        Xi_te=normalize(Xi_te+Xi_f,norm='l2').astype(np.float32)
        del Xv_f,Xi_f; gc.collect()
    extra_te=build_extra(X_test_p,CFG,flip=False)
    if CFG.flip_test and extra_te:
        extra_te_f=build_extra(X_test_p,CFG,flip=True)
        for k in extra_te:
            extra_te[k]=normalize(extra_te[k]+extra_te_f[k],norm='l2').astype(np.float32)
        del extra_te_f; gc.collect()
    X_dict_te={'vlad':Xv_te,'ifv':Xi_te}
    for name,Xf in extra_te.items():
        sc=scalers.get(name)
        if sc: X_dict_te[name]=sc.transform(Xf).astype(np.float32)
    save_npz('features_test.npz',**X_dict_te)

print('\n📐 Chiều vector TEST:')
for k,v in X_dict_te.items():
    print(f'   {k:>6s}: {v.shape}')

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
import inspect # Import inspect module

def train_best_classifier_for_feature(X, y, cfg, feature_name, candidate_models_info):
    """
    Trains multiple candidate classifiers for a given feature using GridSearchCV
    and returns the best performing one. This function does NOT save clfs.
    """
    best_overall_score = -1
    best_overall_estimator = None
    best_overall_params = {}

    print(f"\n─── Training for {feature_name.upper()} (dim={X.shape[1]}) ───")

    # Special case for HOG as per original notebook, using fixed LinearSVC C=10.0
    if feature_name.lower() == 'hog':
        print("  Using fixed LinearSVC (C=10.0) for HOG as per config.")
        clf = LinearSVC(C=10.0, max_iter=20000, dual=True, random_state=cfg.random_state)
        cv_splitter = StratifiedKFold(n_splits=cfg.grid_cv_splits, shuffle=True, random_state=cfg.random_state)
        scores = cross_val_score(clf, X, y, cv=cv_splitter, scoring='accuracy', n_jobs=cfg.grid_n_jobs)
        mean_score = np.mean(scores)
        print(f"    HOG (LinearSVC C=10.0) CV acc={mean_score:.4f}")
        clf.fit(X, y) # Fit on full data for the final estimator
        return clf, {'C': 10.0}, mean_score

    for estimator_class, base_params, param_grid in candidate_models_info:
        # Prepare estimator parameters, adding max_iter conditionally
        current_init_params = base_params.copy()
        current_init_params['random_state'] = cfg.random_state

        if 'max_iter' in inspect.signature(estimator_class.__init__).parameters:
            current_init_params['max_iter'] = 20000

        try:
            estimator = estimator_class(**current_init_params)
            print(f"  Trying {estimator_class.__name__} with params {current_init_params}...")
        except TypeError as e:
            # This block is now more specifically for handling cases where `current_init_params`
            # might still contain incompatible arguments, though `max_iter` is handled above.
            # For example, if 'dual' was passed to LogisticRegression with an incompatible solver.
            # Given the `candidate_models` are well-defined now, this block might be rarely hit.
            if estimator_class == LogisticRegression and 'dual' in current_init_params:
                print(f"  Error initializing {estimator_class.__name__} with {e}. Retrying without 'dual' param.")
                temp_params = {k:v for k,v in current_init_params.items() if k != 'dual'}
                estimator = estimator_class(**temp_params)
                print(f"  Trying {estimator_class.__name__} with adjusted params {temp_params}...")
            else:
                print(f"  Unhandled TypeError for {estimator_class.__name__}: {e}")
                raise # Re-raise if it's not the expected case we can recover from

        cv = StratifiedKFold(n_splits=cfg.grid_cv_splits, shuffle=True, random_state=cfg.random_state)
        gs = GridSearchCV(estimator, param_grid, cv=cv, scoring='accuracy', n_jobs=cfg.grid_n_jobs, verbose=0)
        gs.fit(X, y)

        print(f"    {estimator_class.__name__} results:")
        for mean_score, params in zip(gs.cv_results_['mean_test_score'], gs.cv_results_['params']):
            print(f"      params={params} cv_acc={mean_score:.4f}")
        print(f"    {estimator_class.__name__} best={gs.best_params_} cv_acc={gs.best_score_:.4f}")

        if gs.best_score_ > best_overall_score:
            best_overall_score = gs.best_score_
            best_overall_estimator = gs.best_estimator_
            best_overall_params = gs.best_params_

    print(f"  Best overall for {feature_name.upper()}: {best_overall_estimator.__class__.__name__} best={best_overall_params} cv_acc={best_overall_score:.4f}")
    return best_overall_estimator, best_overall_params, best_overall_score

# Initialize or load clfs dictionary
if exists('clfs.pkl'):
    clfs = load_pkl('clfs.pkl')
else:
    clfs = {}

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import LinearSVC, SVC # Ensure SVC is imported here as well
from sklearn.linear_model import LogisticRegression
import inspect # Import inspect module

def train_best_classifier_for_feature(X, y, cfg, feature_name, candidate_models_info):
    """
    Trains multiple candidate classifiers for a given feature using GridSearchCV
    and returns the best performing one. This function does NOT save clfs.
    """
    best_overall_score = -1
    best_overall_estimator = None
    best_overall_params = {}

    print(f"\n─── Training for {feature_name.upper()} (dim={X.shape[1]}) ───")

    # Removed special case for HOG. All features will now use candidate_models_info.

    for estimator_class, base_params, param_grid in candidate_models_info:
        # Prepare estimator parameters, adding max_iter conditionally
        current_init_params = base_params.copy()
        current_init_params['random_state'] = cfg.random_state

        if 'max_iter' in inspect.signature(estimator_class.__init__).parameters:
            current_init_params['max_iter'] = 20000

        try:
            estimator = estimator_class(**current_init_params)
            print(f"  Trying {estimator_class.__name__} with params {current_init_params}...")
        except TypeError as e:
            # This block is now more specifically for handling cases where `current_init_params`
            # might still contain incompatible arguments, though `max_iter` is handled above.
            # For example, if 'dual' was passed to LogisticRegression with an incompatible solver.
            # Given the `candidate_models` are well-defined now, this block might be rarely hit.
            if estimator_class == LogisticRegression and 'dual' in current_init_params:
                print(f"  Error initializing {estimator_class.__name__} with {e}. Retrying without 'dual' param.")
                temp_params = {k:v for k,v in current_init_params.items() if k != 'dual'}
                estimator = estimator_class(**temp_params)
                print(f"  Trying {estimator_class.__name__} with adjusted params {temp_params}...")
            else:
                print(f"  Unhandled TypeError for {estimator_class.__name__}: {e}")
                raise # Re-raise if it's not the expected case we can recover from

        cv = StratifiedKFold(n_splits=cfg.grid_cv_splits, shuffle=True, random_state=cfg.random_state)
        gs = GridSearchCV(estimator, param_grid, cv=cv, scoring='accuracy', n_jobs=cfg.grid_n_jobs, verbose=0)
        gs.fit(X, y)

        print(f"    {estimator_class.__name__} results:")
        for mean_score, params in zip(gs.cv_results_['mean_test_score'], gs.cv_results_['params']):
            print(f"      params={params} cv_acc={mean_score:.4f}")
        print(f"    {estimator_class.__name__} best={gs.best_params_} cv_acc={gs.best_score_:.4f}")

        if gs.best_score_ > best_overall_score:
            best_overall_score = gs.best_score_
            best_overall_estimator = gs.best_estimator_
            best_overall_params = gs.best_params_

    print(f"  Best overall for {feature_name.upper()}: {best_overall_estimator.__class__.__name__} best={best_overall_params} cv_acc={best_overall_score:.4f}")
    return best_overall_estimator, best_overall_params, best_overall_score

# Initialize or load clfs dictionary
if exists('clfs.pkl'):
    clfs = load_pkl('clfs.pkl')
else:
    clfs = {}

## Cell 18.1 — Huấn luyện SVM cho từng đặc trưng

Bây giờ chúng ta sẽ huấn luyện LinearSVC cho từng bộ đặc trưng (VLAD, IFV, HOG, LBP, GLCM, GIST) bằng `GridSearchCV` để tìm siêu tham số `C` tốt nhất.

In [ ]:
print(f'\n─── VLAD (dim={X_dict_tr["vlad"].shape[1]}) ───')
clf,bp,bcv=train_svc(X_dict_tr['vlad'],ytr,CFG,'vlad')
clfs['vlad']={'est':clf,'best_params':bp,'best_cv':bcv}
print(f'    best={bp}  cv_acc={bcv}')

In [ ]:
print(f'\n─── IFV (dim={X_dict_tr["ifv"].shape[1]}) ───')
clf,bp,bcv=train_svc(X_dict_tr['ifv'],ytr,CFG,'ifv')
clfs['ifv']={'est':clf,'best_params':bp,'best_cv':bcv}
print(f'    best={bp}  cv_acc={bcv}')

In [ ]:
if 'hog' in X_dict_tr:
    print(f'\n─── HOG (dim={X_dict_tr["hog"].shape[1]}) ───')
    clf,bp,bcv=train_svc(X_dict_tr['hog'],ytr,CFG,'hog')
    clfs['hog']={'est':clf,'best_params':bp,'best_cv':bcv}
    print(f'    best={bp}  cv_acc={bcv}')
else:
    print('\n─── HOG: Not used (cfg.use_hog is False) ───')

In [ ]:
if 'lbp' in X_dict_tr:
    print(f'\n─── LBP (dim={X_dict_tr["lbp"].shape[1]}) ───')
    clf,bp,bcv=train_svc(X_dict_tr['lbp'],ytr,CFG,'lbp')
    clfs['lbp']={'est':clf,'best_params':bp,'best_cv':bcv}
    print(f'    best={bp}  cv_acc={bcv}')
else:
    print('\n─── LBP: Not used (cfg.use_lbp is False) ───')

In [ ]:
if 'glcm' in X_dict_tr:
    print(f'\n─── GLCM Haralick (dim={X_dict_tr["glcm"].shape[1]}) ───')
    clf,bp,bcv=train_svc(X_dict_tr['glcm'],ytr,CFG,'glcm')
    clfs['glcm']={'est':clf,'best_params':bp,'best_cv':bcv}
    print(f'    best={bp}  cv_acc={bcv}')
else:
    print('\n─── GLCM: Not used (cfg.use_glcm is False) ───')

In [ ]:
if 'gist' in X_dict_tr:
    print(f'\n─── GIST (dim={X_dict_tr["gist"].shape[1]}) ───')
    clf,bp,bcv=train_svc(X_dict_tr['gist'],ytr,CFG,'gist')
    clfs['gist']={'est':clf,'best_params':bp,'best_cv':bcv}
    print(f'    best={bp}  cv_acc={bcv}')
else:
    print('\n─── GIST: Not used (cfg.use_gist is False) ───')

In [ ]:
# The clfs dictionary is now saved incrementally after each feature's training.
# This cell only prints the status of the models.

print('\n✅ Models:')
for name,v in clfs.items():
    print(f'   {name:>6s}: {v["best_params"]}  cv={v["best_cv"]:.4f}')

# Kiểm tra các bộ phân loại đã lưu
print('\n── Kiểm tra các bộ phân loại đã lưu ──')
expected_features = ['vlad', 'ifv'] # These are always used
if CFG.use_hog: expected_features.append('hog')
if CFG.use_lbp: expected_features.append('lbp')
if CFG.use_glcm: expected_features.append('glcm')
if CFG.use_gist: expected_features.append('gist')

found_features = list(clfs.keys())
missing_features = [f for f in expected_features if f not in found_features]
extra_features = [f for f in found_features if f not in expected_features]

if not missing_features and not extra_features:
    print('   ✅ Tất cả các bộ phân loại đặc trưng mong muốn đã được lưu.')
elif not missing_features and extra_features:
    print(f'   ⚠️  Tìm thấy các bộ phân loại không mong muốn: {extra_features}')
elif missing_features:
    print(f'   ❌ Thiếu các bộ phân loại: {missing_features}')
else:
    print(f'   ❌ Thiếu các bộ phân loại: {missing_features}')
    print(f'   ⚠️  Tìm thấy các bộ phân loại không mong muốn: {extra_features}')

---
## Cell 19 — OOF + Nelder-Mead weight search

In [ ]:
def decision_scores(clf, X):
    s=clf.decision_function(X)
    if s.ndim==1: s=np.vstack([-s,s]).T
    return s.astype(np.float32)

def build_oof_scores(clfs_dict, X_dict, y, cfg):
    skf=StratifiedKFold(n_splits=cfg.oof_splits,shuffle=True,random_state=cfg.random_state)
    n_cl=len(np.unique(y))
    oof={name:np.zeros((len(y),n_cl),dtype=np.float32) for name in clfs_dict}
    for fold,(tr,va) in enumerate(skf.split(np.zeros(len(y)),y),1):
        for name,proto in clfs_dict.items():
            clf_f=LinearSVC(**proto.get_params())
            clf_f.fit(X_dict[name][tr],y[tr])
            oof[name][va]=decision_scores(clf_f,X_dict[name][va])
        print(f'  [OOF] fold {fold}/{cfg.oof_splits}')
    return oof

def weight_search_nelder(oof_scores, y_true, cfg):
    names=list(oof_scores.keys())
    stacked=np.stack([oof_scores[n] for n in names],0)
    def neg_acc(w):
        fused=np.einsum('i,ijk->jk',np.abs(w),stacked)
        return -(np.argmax(fused,1)==y_true).mean()
    res=minimize(neg_acc,x0=np.ones(len(names)),method='Nelder-Mead',
                 options={'maxiter':cfg.nm_maxiter,'xatol':cfg.nm_xatol,'fatol':1e-5})
    w_opt=np.abs(res.x)
    return {names[i]:float(w_opt[i]) for i in range(len(names))}, float(-res.fun)

if exists('weights.pkl'):
    weights=load_pkl('weights.pkl')
else:
    best_clfs={name:clfs[name]['est'] for name in clfs}
    print('── OOF scoring ──')
    oof_scores=build_oof_scores(best_clfs,X_dict_tr,ytr,CFG)
    print('── Nelder-Mead optimization ──')
    weights,oof_acc=weight_search_nelder(oof_scores,ytr,CFG)
    print(f'  OOF acc = {oof_acc:.4f}')
    save_pkl(weights,'weights.pkl')

print('\n✅ Weights (sorted):')
for k,v in sorted(weights.items(),key=lambda x:-x[1]):
    print(f'   {k:>6s}: {v:.4f}')

---
## Cell 20 — Đánh giá TEST & lưu kết quả

In [ ]:
print('── Base accuracy (từng đặc trưng) ──')
scores_te={}; base_accs={}
for name,Xf in X_dict_te.items():
    if name not in clfs: continue
    yp=clfs[name]['est'].predict(Xf)
    acc=accuracy_score(y_test,yp)
    base_accs[name]=acc
    print(f'   {name.upper():>6s}  acc = {acc:.4f}')
    scores_te[name]=decision_scores(clfs[name]['est'],Xf)

fused=None
for name in sorted(scores_te,key=lambda n:-weights.get(n,0)):
    w=float(weights.get(name,0.0))
    fused=scores_te[name]*w if fused is None else fused+scores_te[name]*w

y_pred=np.argmax(fused,1)
acc_ens=accuracy_score(y_test,y_pred)
cm=confusion_matrix(y_test,y_pred)
report=classification_report(y_test,y_pred,target_names=classes)

print(f'\n🎯 Ensemble accuracy = {acc_ens:.4f}')
print(f'   Weights = {weights}')
print('\n'+report)

results={'acc_ens':acc_ens,'base_accs':base_accs,'weights':weights,
         'cm':cm,'report':report,'classes':classes,'y_pred':y_pred,'y_test':y_test}
save_pkl(results,'results.pkl')

## Biểu đồ Confusion Matrix (phần trăm) của Ensemble trên tập Test

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print('── Biểu đồ Confusion Matrix (phần trăm) của Ensemble (tập TEST) ──')

# Lấy kết quả từ results.pkl
r = load_pkl('results.pkl')
cm = r['cm']
acc_ens = r['acc_ens']
classes = r['classes']

# Chuyển đổi ma trận nhầm lẫn sang dạng phần trăm
# Chia mỗi hàng cho tổng của hàng đó (tức là tổng số mẫu thực tế của lớp đó)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Vẽ biểu đồ
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_percent, annot=True, fmt='.1%', cmap='Blues', cbar=True, ax=ax,
            xticklabels=classes, yticklabels=classes)

ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thực tế')
ax.set_title(f'Confusion Matrix (phần trăm) Ensemble — TEST (Accuracy: {acc_ens:.4f})', pad=12)

plt.tight_layout()
save_fig(fig, 'confusion_matrix_ensemble_percent_test.png')
plt.show()

---
## Cell 21 — 📊 Confusion Matrix

In [ ]:
fig,ax=plt.subplots(figsize=(10,8))
im=ax.imshow(cm,cmap='Blues')
fig.colorbar(im,ax=ax,shrink=0.82)
ticks=np.arange(len(classes))
ax.set_xticks(ticks); ax.set_xticklabels(classes,rotation=45,ha='right',fontsize=8)
ax.set_yticks(ticks); ax.set_yticklabels(classes,fontsize=8)
thr=cm.max()/2
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j,i,cm[i,j],ha='center',va='center',fontsize=7,
                color='white' if cm[i,j]>thr else 'black')
ax.set_title(f'Confusion Matrix — Ensemble  acc={acc_ens:.4f}',pad=12)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
save_fig(fig,'confusion_matrix.png')
plt.show()

---
## Cell 22 — 📊 So sánh accuracy từng đặc trưng

In [ ]:
r=load_pkl('results.pkl')
ba=r['base_accs']; ae=r['acc_ens']
labels_b=list(ba.keys())+['ENSEMBLE']
vals_b=list(ba.values())+[ae]
cols_b=['#378ADD']*len(ba)+['#1D9E75']

fig,ax=plt.subplots(figsize=(9,4))
bars=ax.barh(labels_b,vals_b,color=cols_b,height=0.5)
for bar,val in zip(bars,vals_b):
    ax.text(val+0.002,bar.get_y()+bar.get_height()/2,
            f'{val:.4f}',va='center',fontsize=9)
ax.set_xlim(0,min(1.0,max(vals_b)+0.06))
ax.axvline(ae,color='#1D9E75',linestyle='--',linewidth=1,alpha=0.6)
ax.set_xlabel('Accuracy'); ax.set_title('Per-feature accuracy vs Ensemble')
plt.tight_layout()
save_fig(fig,'accuracy_comparison.png')
plt.show()

---
## Cell 23 — 🔄 Restore session (chạy ngay sau kernel restart)

Khi bị ngắt phiên, chạy theo thứ tự:
**Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 (mount Drive) → Cell 23**

In [ ]:
def restore_session():
    global pca, kmeans, gmm, scalers, sift, X_raw, X_pca_proj
    global X_train_p, X_test_p, y_train, y_test, classes
    global X_dict_tr, X_dict_te, ytr
    global clfs, weights, results

    sift = cv2.SIFT_create()

    if exists('labels.npz'):
        d=load_npz('labels.npz')
        X_train_p=d['X_train_p']; X_test_p=d['X_test_p']
        y_train=d['y_train'];     y_test=d['y_test']
        classes=list(d['classes'])
        print(f'  classes={classes}')

    if exists('pca_raw_descriptors.npz'):
        d=load_npz('pca_raw_descriptors.npz')
        X_raw=d['X_raw']

    if exists('pca.pkl'):    pca    = load_pkl('pca.pkl');    CFG.pca_dim=pca.n_components_
    if exists('kmeans.pkl'): kmeans = load_pkl('kmeans.pkl'); CFG.vlad_k=kmeans.n_clusters
    if exists('gmm.pkl'):    gmm    = load_pkl('gmm.pkl')
    scalers = load_pkl('scalers_train.pkl') if exists('scalers_train.pkl') else {}

    # Rebuild X_pca_proj nếu cần (cho biểu đồ elbow)
    if 'pca' in dir() and 'X_raw' in dir():
        rng=np.random.default_rng(CFG.random_state)
        take=min(CFG.vlad_max_desc,len(X_raw))
        idx=rng.choice(len(X_raw),take,replace=False)
        X_pca_proj=pca.transform(X_raw[idx]).astype(np.float32)

    if exists('features_train.npz'):
        d=load_npz('features_train.npz')
        ytr=d['ytr']; X_dict_tr={k:d[k] for k in d if k!='ytr'}

    if exists('features_test.npz'):
        d=load_npz('features_test.npz')
        X_dict_te={k:d[k] for k in d}

    if exists('clfs.pkl'):    clfs    = load_pkl('clfs.pkl')
    if exists('weights.pkl'): weights = load_pkl('weights.pkl')
    if exists('results.pkl'): results = load_pkl('results.pkl')

    print(f'\n  CFG.pca_dim = {CFG.pca_dim}')
    print(f'  CFG.vlad_k  = {CFG.vlad_k}')
    print('\n✅ Session restored!')
    checkpoint_status()

restore_session()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print('── Biểu đồ Confusion Matrix cho VLAD (tập TEST) ──')

# Lấy bộ phân loại VLAD và dữ liệu TEST
vlad_clf = clfs['vlad']['est']
X_vlad_te = X_dict_te['vlad']

# Dự đoán trên tập TEST VLAD
y_pred_vlad_te = vlad_clf.predict(X_vlad_te)

# Tính ma trận nhầm lẫn
cm_vlad = confusion_matrix(y_test, y_pred_vlad_te)

# Tính độ chính xác cơ sở của VLAD trên tập TEST (chỉ để hiển thị)
acc_vlad_te = (y_pred_vlad_te == y_test).mean()

# Vẽ biểu đồ
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_vlad, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax,
            xticklabels=classes, yticklabels=classes)

ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thực tế')
ax.set_title(f'Confusion Matrix cho VLAD — TEST (Accuracy: {acc_vlad_te:.4f})', pad=12)

plt.tight_layout()
save_fig(fig, 'confusion_matrix_vlad_test.png')
plt.show()

In [ ]:
# Cell huấn luyện cho đặc trưng VLAD
candidate_models = [
    (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']}), # Changed to SVC RBF
    (LogisticRegression, {'solver': 'liblinear', 'dual': False}, {'C': [0.1, 1.0, 10.0]}),
    (RandomForestClassifier, {}, {'n_estimators': [100, 200], 'max_depth': [5, 10]}),
    (GradientBoostingClassifier, {}, {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.05], 'max_depth': [3, 5]})
]

clf, best_params, best_cv_score = train_best_classifier_for_feature(
    X_dict_tr['vlad'], ytr, CFG, 'vlad', candidate_models
)
clfs['vlad'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
save_pkl(clfs, 'clfs.pkl')

print(f"\n  VLAD Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")

In [ ]:
# Cell huấn luyện cho đặc trưng IFV
candidate_models = [
    (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']}), # Changed to SVC RBF
    (LogisticRegression, {'solver': 'liblinear', 'dual': False}, {'C': [0.1, 1.0, 10.0]}),
    (RandomForestClassifier, {}, {'n_estimators': [100, 200], 'max_depth': [5, 10]}),
    (GradientBoostingClassifier, {}, {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.05], 'max_depth': [3, 5]})
]

clf, best_params, best_cv_score = train_best_classifier_for_feature(
    X_dict_tr['ifv'], ytr, CFG, 'ifv', candidate_models
)
clfs['ifv'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
save_pkl(clfs, 'clfs.pkl')

print(f"\n  IFV Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")

In [ ]:
# Cell huấn luyện cho đặc trưng HOG
if 'hog' in X_dict_tr:
    candidate_models = [
        (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']}), # Changed to SVC RBF
        (LogisticRegression, {'solver': 'liblinear', 'dual': False}, {'C': [0.1, 1.0, 10.0]}),
        (RandomForestClassifier, {}, {'n_estimators': [100, 200], 'max_depth': [5, 10]}),
        (GradientBoostingClassifier, {}, {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.05], 'max_depth': [3, 5]})
    ]

    clf, best_params, best_cv_score = train_best_classifier_for_feature(
        X_dict_tr['hog'], ytr, CFG, 'hog', candidate_models
    )
    clfs['hog'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
    save_pkl(clfs, 'clfs.pkl')
    print(f"\n  HOG Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")
elser:
    print('\n─── HOG: Not used (cfg.use_hog is False) ───')
    # If HOG is not used, ensure it's not in clfs if it was previously loaded from checkpoint
    if 'hog' in clfs: del clfs['hog']; save_pkl(clfs, 'clfs.pkl')

In [ ]:
# Cell huấn luyện cho đặc trưng LBP
if 'lbp' in X_dict_tr:
    candidate_models = [
        (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']}), # Changed to SVC RBF
        (LogisticRegression, {'solver': 'liblinear', 'dual': False}, {'C': [0.1, 1.0, 10.0]}),
        (RandomForestClassifier, {}, {'n_estimators': [100, 200], 'max_depth': [5, 10]}),
        (GradientBoostingClassifier, {}, {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.05], 'max_depth': [3, 5]})
    ]

    clf, best_params, best_cv_score = train_best_classifier_for_feature(
        X_dict_tr['lbp'], ytr, CFG, 'lbp', candidate_models
    )
    clfs['lbp'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
    save_pkl(clfs, 'clfs.pkl')
    print(f"\n  LBP Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")
elser:
    print('\n─── LBP: Not used (cfg.use_lbp is False) ───')
    if 'lbp' in clfs: del clfs['lbp']; save_pkl(clfs, 'clfs.pkl')

In [ ]:
# Cell huấn luyện cho đặc trưng GLCM Haralick
if 'glcm' in X_dict_tr:
    candidate_models = [
        (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']}), # Changed to SVC RBF
        (LogisticRegression, {'solver': 'liblinear', 'dual': False}, {'C': [0.1, 1.0, 10.0]}),
        (RandomForestClassifier, {}, {'n_estimators': [100, 200], 'max_depth': [5, 10]}),
        (GradientBoostingClassifier, {}, {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.05], 'max_depth': [3, 5]})
    ]

    clf, best_params, best_cv_score = train_best_classifier_for_feature(
        X_dict_tr['glcm'], ytr, CFG, 'glcm', candidate_models
    )
    clfs['glcm'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
save_pkl(clfs, 'clfs.pkl')
    print(f"\n  GLCM Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")
elser:
    print('\n─── GLCM: Not used (cfg.use_glcm is False) ───')
    if 'glcm' in clfs: del clfs['glcm']; save_pkl(clfs, 'clfs.pkl')

In [ ]:
# Cell huấn luyện cho đặc trưng GIST
if 'gist' in X_dict_tr:
    candidate_models = [
        (SVC, {'kernel': 'rbf'}, {'C': [1.0, 10.0], 'gamma': ['scale', 'auto']}), # Changed to SVC RBF
        (LogisticRegression, {'solver': 'liblinear', 'dual': False}, {'C': [0.1, 1.0, 10.0]}),
        (RandomForestClassifier, {}, {'n_estimators': [100, 200], 'max_depth': [5, 10]}),
        (GradientBoostingClassifier, {}, {'n_estimators': [100, 200], 'learning_rate': [0.1, 0.05], 'max_depth': [3, 5]})
    ]

    clf, best_params, best_cv_score = train_best_classifier_for_feature(
        X_dict_tr['gist'], ytr, CFG, 'gist', candidate_models
    )
    clfs['gist'] = {'est': clf, 'best_params': best_params, 'best_cv': best_cv_score}
save_pkl(clfs, 'clfs.pkl')
    print(f"\n  GIST Model saved: best={best_params} cv_acc={best_cv_score:.4f}. Current clfs keys: {list(clfs.keys())}")
elser:
    print('\n─── GIST: Not used (cfg.use_gist is False) ───')
    if 'gist' in clfs: del clfs['gist']; save_pkl(clfs, 'clfs.pkl')

---
## Cell 24 — 📋 Checkpoint status

In [ ]:
checkpoint_status()